In [0]:
# 1. Criação do schema (banco de dados) para a camada Bronze no catálogo 'mvp'
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.bronze")

# 2. Dicionário mapeando o nome da tabela e o respectivo arquivo de origem
arquivos_olist = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "customers": "olist_customers_dataset.csv"
}

# 3. Caminho base dos arquivos nos Volumes (conforme seu print)
caminho_base = "/Volumes/mvp/default/mvp/"

# 4. Loop de ingestão: Lendo cada CSV e salvando como tabela Delta
for tabela, arquivo in arquivos_olist.items():
    print(f"Iniciando ingestão do arquivo: {arquivo} ...")
    
    # Leitura do CSV bruto
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("escape", '"') \
        .load(f"{caminho_base}{arquivo}")
    
    # Escrita no formato Delta dentro do schema bronze
    df_raw.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"mvp.bronze.{tabela}")
    
    print(f"Sucesso! Tabela mvp.bronze.{tabela} criada.\n")

# 5. Verifica e exibe as tabelas criadas no schema bronze
display(spark.sql("SHOW TABLES IN mvp.bronze"))

Iniciando ingestão do arquivo: olist_orders_dataset.csv ...
Sucesso! Tabela mvp.bronze.orders criada.

Iniciando ingestão do arquivo: olist_order_items_dataset.csv ...
Sucesso! Tabela mvp.bronze.order_items criada.

Iniciando ingestão do arquivo: olist_order_reviews_dataset.csv ...
Sucesso! Tabela mvp.bronze.reviews criada.

Iniciando ingestão do arquivo: olist_products_dataset.csv ...
Sucesso! Tabela mvp.bronze.products criada.

Iniciando ingestão do arquivo: olist_customers_dataset.csv ...
Sucesso! Tabela mvp.bronze.customers criada.



database,tableName,isTemporary
bronze,customers,false
bronze,order_items,false
bronze,orders,false
bronze,products,false
bronze,reviews,false


In [0]:
from pyspark.sql.functions import col, coalesce, lit, expr

# 1. Criação do schema para a camada Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.silver")

print("Iniciando o processamento da camada Silver...\n")

# 2. Tratamento da tabela ORDERS
print("Limpando e formatando: orders")
df_orders = spark.read.table("mvp.bronze.orders").dropDuplicates()

# Usando try_cast para lidar com dados que vieram em formato inválido
colunas_data_orders = [
    "order_purchase_timestamp", 
    "order_approved_at", 
    "order_delivered_carrier_date", 
    "order_delivered_customer_date", 
    "order_estimated_delivery_date"
]
for c in colunas_data_orders:
    df_orders = df_orders.withColumn(c, expr(f"try_cast({c} as timestamp)"))
    
df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp.silver.orders")


# 3. Tratamento da tabela ORDER_ITEMS
print("Limpando e formatando: order_items")
df_items = spark.read.table("mvp.bronze.order_items").dropDuplicates()
df_items = df_items.withColumn("shipping_limit_date", expr("try_cast(shipping_limit_date as timestamp)"))
df_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp.silver.order_items")


# 4. Tratamento da tabela REVIEWS
print("Limpando e formatando: reviews")
df_reviews = spark.read.table("mvp.bronze.reviews").dropDuplicates()
# O try_cast converte datas válidas e joga "null" onde houver textos digitados incorretamente
df_reviews = df_reviews.withColumn("review_creation_date", expr("try_cast(review_creation_date as timestamp)")) \
                       .withColumn("review_answer_timestamp", expr("try_cast(review_answer_timestamp as timestamp)")) \
                       .withColumn("review_comment_message", coalesce(col("review_comment_message"), lit("Sem comentário"))) \
                       .withColumn("review_comment_title", coalesce(col("review_comment_title"), lit("Sem título")))
df_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp.silver.reviews")


# 5. Tratamento da tabela PRODUCTS
print("Limpando e formatando: products")
df_products = spark.read.table("mvp.bronze.products").dropDuplicates()
df_products = df_products.withColumn("product_category_name", coalesce(col("product_category_name"), lit("nao_informado")))
df_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp.silver.products")


# 6. Tratamento da tabela CUSTOMERS
print("Limpando e formatando: customers")
df_customers = spark.read.table("mvp.bronze.customers").dropDuplicates()
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp.silver.customers")

print("\nProcessamento da camada Silver concluído com sucesso!")

# 7. Verifica e exibe as tabelas criadas no schema silver
display(spark.sql("SHOW TABLES IN mvp.silver"))

Iniciando o processamento da camada Silver...

Limpando e formatando: orders
Limpando e formatando: order_items
Limpando e formatando: reviews
Limpando e formatando: products
Limpando e formatando: customers

Processamento da camada Silver concluído com sucesso!


database,tableName,isTemporary
silver,customers,false
silver,order_items,false
silver,orders,false
silver,products,false
silver,reviews,false


In [0]:
print("Iniciando a modelagem da camada Gold...\n")

# 1. Criação do schema Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.gold")

# 2. Criando Dimensão de Clientes
print("Criando dim_clientes...")
spark.sql("""
    CREATE OR REPLACE TABLE mvp.gold.dim_clientes AS
    SELECT 
        customer_id,
        customer_unique_id,
        customer_city,
        customer_state
    FROM mvp.silver.customers
""")

# 3. Criando Dimensão de Produtos
print("Criando dim_produtos...")
spark.sql("""
    CREATE OR REPLACE TABLE mvp.gold.dim_produtos AS
    SELECT 
        product_id,
        product_category_name
    FROM mvp.silver.products
""")

# 4. Criando Fato Vendas
print("Criando fato_vendas...")
spark.sql("""
    CREATE OR REPLACE TABLE mvp.gold.fato_vendas AS
    SELECT 
        i.order_id,
        i.order_item_id,
        i.product_id,
        o.customer_id,
        i.price,
        i.freight_value,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        -- Cálculo de dias de atraso (se > 0, atrasou)
        datediff(o.order_delivered_customer_date, o.order_estimated_delivery_date) AS dias_atraso,
        r.review_score
    FROM mvp.silver.order_items i
    INNER JOIN mvp.silver.orders o ON i.order_id = o.order_id
    -- Agrupando reviews para evitar duplicação caso um pedido tenha múltiplas avaliações
    LEFT JOIN (
        SELECT order_id, MIN(review_score) AS review_score 
        FROM mvp.silver.reviews 
        GROUP BY order_id
    ) r ON i.order_id = r.order_id
    WHERE o.order_status = 'delivered'
""")

print("\nModelagem Gold concluída com sucesso!")
display(spark.sql("SHOW TABLES IN mvp.gold"))

Iniciando a modelagem da camada Gold...

Criando dim_clientes...
Criando dim_produtos...
Criando fato_vendas...

Modelagem Gold concluída com sucesso!


database,tableName,isTemporary
gold,dim_clientes,false
gold,dim_produtos,false
gold,fato_vendas,false


In [0]:
print("--- 0. Verificação de Qualidade (Quality Check) ---")
# Verifica se há pedidos sem cliente ou sem preço associado na tabela Fato
display(spark.sql("""
    SELECT 
        COUNT(*) as total_registros,
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) as clientes_nulos,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) as precos_nulos
    FROM mvp.gold.fato_vendas
"""))

print("\n--- 1. Faturamento: Top 10 Categorias ---")
# Responde à Pergunta 2 (Categorias com maior faturamento bruto)
display(spark.sql("""
    SELECT 
        p.product_category_name AS categoria,
        ROUND(SUM(f.price), 2) AS faturamento_total_brl
    FROM mvp.gold.fato_vendas f
    JOIN mvp.gold.dim_produtos p ON f.product_id = p.product_id
    GROUP BY p.product_category_name
    ORDER BY faturamento_total_brl DESC
    LIMIT 10
"""))

print("\n--- 2. Logística, Satisfação e Gargalos ---")
# Responde às Perguntas 1 e 4 (Impacto do atraso nas notas e % de notas 1)
display(spark.sql("""
    SELECT 
        CASE 
            WHEN dias_atraso > 0 THEN 'Atrasado'
            ELSE 'No Prazo / Adiantado'
        END AS status_logistico,
        COUNT(DISTINCT order_id) AS total_pedidos,
        ROUND(AVG(review_score), 2) AS nota_media_cliente,
        SUM(CASE WHEN review_score = 1 THEN 1 ELSE 0 END) AS qtd_notas_1,
        ROUND((SUM(CASE WHEN review_score = 1 THEN 1 ELSE 0 END) / COUNT(DISTINCT order_id)) * 100, 2) AS percentual_notas_1
    FROM mvp.gold.fato_vendas
    WHERE review_score IS NOT NULL
    GROUP BY status_logistico
"""))

print("\n--- 3. Geografia e Custos ---")
# Responde à Pergunta 3 (Volume de pedidos e frete médio por estado)
display(spark.sql("""
    SELECT 
        c.customer_state AS estado_uf,
        COUNT(DISTINCT f.order_id) AS volume_pedidos,
        ROUND(AVG(f.freight_value), 2) AS frete_medio_brl
    FROM mvp.gold.fato_vendas f
    JOIN mvp.gold.dim_clientes c ON f.customer_id = c.customer_id
    GROUP BY c.customer_state
    ORDER BY volume_pedidos DESC
"""))

--- 0. Verificação de Qualidade (Quality Check) ---


total_registros,clientes_nulos,precos_nulos
110197,0,0



--- 1. Faturamento: Top 10 Categorias ---


categoria,faturamento_total_brl
beleza_saude,1233131.72
relogios_presentes,1166176.98
cama_mesa_banho,1023434.76
esporte_lazer,954852.55
informatica_acessorios,888724.61
moveis_decoracao,711927.69
utilidades_domesticas,615628.69
cool_stuff,610204.1
automotivo,578966.65
brinquedos,471286.48



--- 2. Logística, Satisfação e Gargalos ---


status_logistico,total_pedidos,nota_media_cliente,qtd_notas_1,percentual_notas_1
No Prazo / Adiantado,89451,4.21,8679,9.7
Atrasado,6381,2.26,3858,60.46



--- 3. Geografia e Custos ---


estado_uf,volume_pedidos,frete_medio_brl
SP,40501,15.12
RJ,12350,20.91
MG,11354,20.63
RS,5345,21.61
PR,4923,20.47
SC,3546,21.51
BA,3256,26.49
DF,2080,21.07
ES,1995,22.03
GO,1957,22.56
